PYDANTIC DATA VALIDATION AND OUTPUT PARSING 

In [1]:
import os
from langchain.chat_models import init_chat_model

In [2]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
model = init_chat_model(model_provider='groq',
                        model = 'llama-3.3-70b-versatile')


In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description = "Title of the movie")
    year: int = Field(description = "Release year of the movie")
    director: str = Field(description = "Director of the movie")
    rating: float = Field(description = "Rating of the movie out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)

In [5]:
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x111cf38c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x111f24440>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'Release year of the movie', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'Rating of the movie out of 10', 'type': 'n

In [6]:
model_with_structure.invoke("Provide the details of the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

In [7]:
model.invoke("Provide me the details of the movie Inception")

AIMessage(content='**Movie Title:** Inception\n**Release Date:** July 16, 2010\n**Director:** Christopher Nolan\n**Genre:** Science Fiction, Action, Thriller\n\n**Plot:**\n\nInception is a mind-bending sci-fi action film that delves into the concept of shared dreaming. The story follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" – planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, including Arthur (Joseph Gordon-Levitt), a point man; Ariadne (Ellen Page), an architect who designs the

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with recommendation details."""
    title: str = Field(..., description = "Title of the movie")
    year: int = Field(..., description = "Release year of the movie")
    director: str = Field(..., description = "Director of the movie")
    rating: float = Field(..., description = "Rating of the movie out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
#include_raw will include the raw response from the model along with the structured output
response = model_with_structure.invoke("Provide the details of the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xmffky8kq', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 290, 'total_tokens': 322, 'completion_time': 0.053536163, 'completion_tokens_details': None, 'prompt_time': 0.016570591, 'prompt_tokens_details': None, 'queue_time': 0.17944209, 'total_time': 0.070106754}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8fc2-9ffb-73c0-a26b-f7c4794aa313-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'xmffky8kq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 32, 

NESTED STRUCTURE IN PYDANTIC

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(default=None, description="Budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide the details of the movie Bahubali")
response

MovieDetails(title='Bahubali', year=2015, cast=[Actor(name='Prabhas', role='Amarendra Baahubali'), Actor(name='Rana Daggubati', role='Bhalla Deva')], genres=['Action', 'Adventure', 'Drama'], budget=250000000.0)

TypedDict

TypedDict provides a simpler alternative using pythons buildin typing, ideal when you dont need runtime validations

In [14]:
from typing_extensions import Annotated, TypedDict

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide the details of the movie Inception")
response

{'director': 'Christopher Nolan',
 'rating': 8.5,
 'title': 'Inception',
 'year': 2010}

In [17]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(default=None, description="Budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide the details of the movie Bahubali")
response

{'budget': 250,
 'cast': [{'name': 'Prabhas', 'role': 'Amarendra Baahubali'},
  {'name': 'Rana Daggubati', 'role': 'Bhallaladeva'},
  {'name': 'Anushka Shetty', 'role': 'Devasena'}],
 'genres': ['Action', 'Adventure', 'Drama'],
 'title': 'Baahubali',
 'year': 2015}

In [18]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

DATA CLASSES

In [19]:
class ContactInfo(BaseModel):
    '''Contact information for a person.'''
    name: str = Field(description="The full name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [22]:
import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

from langchain.chat_models import init_chat_model
model = init_chat_model(model_provider='groq',
                        model = 'llama-3.3-70b-versatile')

from langchain.agents import create_agent
agent = create_agent(model=model,
                     response_format=ContactInfo)

result = agent.invoke({
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that provides contact information for a person."},
        {"role": "user", "content": "Provide the contact information for John Doe."}
    ]
})
print(result)
result['structured_response']

{'messages': [SystemMessage(content='You are a helpful assistant that provides contact information for a person.', additional_kwargs={}, response_metadata={}, id='3c1c289f-ec80-4517-810d-20b922d75a28'), HumanMessage(content='Provide the contact information for John Doe.', additional_kwargs={}, response_metadata={}, id='9a2c3954-9245-411b-990b-e81631451403'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ycfa9v0ax', 'function': {'arguments': '{"email":"john.doe@example.com","name":"John Doe","phone":"123-456-7890"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 290, 'total_tokens': 324, 'completion_time': 0.064532304, 'completion_tokens_details': None, 'prompt_time': 0.014531732, 'prompt_tokens_details': None, 'queue_time': 0.020259768, 'total_time': 0.079064036}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'to

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

In [23]:
#using Dataclasses
from dataclasses import dataclass, field

@dataclass
class ContactInfo:
    '''Contact information for a person.'''
    name: str = field(metadata={"description": "The full name of the person"})
    email: str = field(metadata={"description": "The email address of the person"})
    phone: str = field(metadata={"description": "The phone number of the person"})



import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

from langchain.chat_models import init_chat_model
model = init_chat_model(model_provider='groq',
                        model = 'llama-3.3-70b-versatile')

from langchain.agents import create_agent
agent = create_agent(model=model,
                     response_format=ContactInfo)

result = agent.invoke({
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that provides contact information for a person."},
        {"role": "user", "content": "Provide the contact information for John Doe."}
    ]
})
print(result)
result['structured_response']

{'messages': [SystemMessage(content='You are a helpful assistant that provides contact information for a person.', additional_kwargs={}, response_metadata={}, id='5eb0aca0-5b22-412c-b877-8999f8da500a'), HumanMessage(content='Provide the contact information for John Doe.', additional_kwargs={}, response_metadata={}, id='96f6d277-42f6-4395-8d4f-7e28e1177e74'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'avgy6tdp0', 'function': {'arguments': '{"email":"johndoe@example.com","name":"John Doe","phone":"1234567890"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 290, 'total_tokens': 323, 'completion_time': 0.079066106, 'completion_tokens_details': None, 'prompt_time': 0.128461775, 'prompt_tokens_details': None, 'queue_time': 0.195814883, 'total_time': 0.207527881}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_

ContactInfo(name='John Doe', email='johndoe@example.com', phone='1234567890')